# PERSUADE Analysis v5: Fluency & Repetition Controls

**Goal:** Determine whether curve-shape differences (low > high) reflect:
1. Baseline fluency/difficulty (perplexity)
2. Length
3. Repetition/template effects
4. Genuine long-range structure

**Primary metrics (all *_128, computed on windows [32,64,128]):**
- `half_life_128`
- `auc_128`
- `log_slope_128`
- `delta_128`
- `slope_early_128` (32→64)
- `slope_late_128` (64→128)

**Controls:**
- `ppl_W32` as baseline fluency
- Repetition metrics: ttr, repeat_bigram_rate, repeat_trigram_rate, distinct2, distinct3

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.integrate import trapezoid
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from pathlib import Path
from collections import Counter
import re

In [ ]:
# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/LRTIA/Results/Persuade")
    DATA_DIR = Path("/content/drive/MyDrive/LRTIA/Data/persuade_clean/cohorts")
    IN_COLAB = True
except:
    BASE_DIR = Path("../results/persuade")
    DATA_DIR = Path("../data/persuade_clean/cohorts")
    IN_COLAB = False

print(f"Results dir: {BASE_DIR}")
print(f"Data dir: {DATA_DIR}")

In [ ]:
# Configuration
WINDOWS_PRIMARY = [32, 64, 128]  # For *_128 metrics
WINDOWS_ALL = [32, 64, 128, 256, 384, 512]

PRIMARY_METRICS = ['half_life_128', 'auc_128', 'log_slope_128', 'delta_128', 'slope_early_128', 'slope_late_128']

COLORS = {
    'score': {'low': '#e74c3c', 'mid': '#f39c12', 'high': '#2ecc71'},
    'score_long': {'low': '#e74c3c', 'mid': '#f39c12', 'high': '#2ecc71'},
    'grade': {6: '#3498db', 9: '#9b59b6', 12: '#e74c3c'},
    'ell': {'ELL': '#e74c3c', 'non_ELL': '#2ecc71'},
}

# Which experiment to run
EXPERIMENT = 'score_long'  # Options: 'score', 'score_long', 'grade', 'ell', 'prompt'

print(f"Running experiment: {EXPERIMENT}")

## 1. Load Data

In [ ]:
# Load existing results
results_path = BASE_DIR / EXPERIMENT / 'essay_level_results.csv'
df = pd.read_csv(results_path)
print(f"Loaded {len(df)} essays from {results_path}")

# Load original cohort for text (needed for repetition metrics)
cohort_files = {
    'score': 'persuade_score_cohort.jsonl',
    'score_long': 'persuade_score_long_cohort.jsonl',
    'grade': 'persuade_grade_cohort.jsonl',
    'ell': 'persuade_ell_cohort.jsonl',
    'prompt': 'persuade_prompt_cohort.jsonl',
}

import json
cohort_path = DATA_DIR / cohort_files[EXPERIMENT]
cohort_texts = {}
with open(cohort_path, 'r') as f:
    for line in f:
        r = json.loads(line)
        cohort_texts[r['essay_id']] = r['text']
print(f"Loaded {len(cohort_texts)} essay texts")

# Set up grouping
if EXPERIMENT in ['score', 'score_long']:
    GROUP_VAR = 'score_bin'
    GROUP_ORDER = ['low', 'mid', 'high']
elif EXPERIMENT == 'grade':
    GROUP_VAR = 'grade'
    GROUP_ORDER = sorted(df['grade'].dropna().unique())
elif EXPERIMENT == 'ell':
    GROUP_VAR = 'ell'
    GROUP_ORDER = ['ELL', 'non_ELL']
else:
    GROUP_VAR = 'prompt_id'
    GROUP_ORDER = df['prompt_id'].value_counts().index.tolist()[:6]

print(f"\nGrouping by: {GROUP_VAR}")
print(f"Groups: {GROUP_ORDER}")
print(f"\nGroup counts:")
print(df[GROUP_VAR].value_counts())

## 2. Compute Primary Metrics (*_128)

In [ ]:
def compute_half_life(ppl_dict, percentile=0.5):
    """Compute half-life from ppl dict."""
    if len(ppl_dict) < 2:
        return np.nan
    items = sorted(ppl_dict.items())
    windows = np.array([x[0] for x in items])
    ppls = np.array([x[1] for x in items])
    
    total_benefit = ppls[0] - ppls[-1]
    if total_benefit <= 0:
        return np.nan
    
    target_ppl = ppls[0] - percentile * total_benefit
    for i in range(len(ppls) - 1):
        if ppls[i] >= target_ppl >= ppls[i + 1]:
            frac = (ppls[i] - target_ppl) / (ppls[i] - ppls[i + 1])
            return windows[i] + frac * (windows[i + 1] - windows[i])
    return windows[-1]


def compute_primary_metrics(row):
    """Compute *_128 metrics using only windows [32, 64, 128]."""
    
    # Extract ppl values for primary windows
    ppls = {}
    for W in WINDOWS_PRIMARY:
        col = f'ppl_W{W}'
        if col in row and pd.notna(row[col]):
            ppls[W] = row[col]
    
    result = {
        'half_life_128': np.nan,
        'auc_128': np.nan,
        'log_slope_128': np.nan,
        'delta_128': np.nan,
        'slope_early_128': np.nan,
        'slope_late_128': np.nan,
    }
    
    if len(ppls) < 3:
        return pd.Series(result)
    
    ppl_32, ppl_64, ppl_128 = ppls[32], ppls[64], ppls[128]
    
    # half_life_128
    result['half_life_128'] = compute_half_life(ppls, 0.5)
    
    # delta_128 = ppl_32 - ppl_128
    result['delta_128'] = ppl_32 - ppl_128
    
    # auc_128: area under delta curve
    windows = np.array([32, 64, 128])
    delta_ppl = np.array([0, ppl_32 - ppl_64, ppl_32 - ppl_128])
    result['auc_128'] = trapezoid(delta_ppl, windows)
    
    # log_slope_128: slope of delta_ppl vs log(window)
    log_windows = np.log(windows)
    slope, _, _, _, _ = stats.linregress(log_windows, delta_ppl)
    result['log_slope_128'] = slope
    
    # slope_early_128: 32→64 (per log-token)
    # delta at 32 = 0, delta at 64 = ppl_32 - ppl_64
    result['slope_early_128'] = (ppl_32 - ppl_64) / (np.log(64) - np.log(32))
    
    # slope_late_128: 64→128 (per log-token)
    result['slope_late_128'] = (ppl_64 - ppl_128) / (np.log(128) - np.log(64))
    
    return pd.Series(result)


# Apply to all rows
print("Computing primary *_128 metrics...")
metrics_128 = df.apply(compute_primary_metrics, axis=1)

# Update existing columns or add new ones
for col in metrics_128.columns:
    df[col] = metrics_128[col]

print(f"\nPrimary metrics computed. Sample:")
print(df[['essay_id', 'score_bin'] + PRIMARY_METRICS].head())

## 3. Compute Repetition Metrics

In [ ]:
def tokenize_words(text):
    """Simple word tokenization."""
    # Lowercase, remove punctuation, split
    text = text.lower()
    words = re.findall(r'\b[a-z]+\b', text)
    return words


def compute_repetition_metrics(text):
    """Compute repetition/template metrics from essay text."""
    words = tokenize_words(text)
    n_words = len(words)
    
    if n_words < 3:
        return {
            'ttr': np.nan,
            'repeat_bigram_rate': np.nan,
            'repeat_trigram_rate': np.nan,
            'distinct2': np.nan,
            'distinct3': np.nan,
        }
    
    # Type-token ratio
    unique_words = len(set(words))
    ttr = unique_words / n_words
    
    # Bigrams
    bigrams = [tuple(words[i:i+2]) for i in range(len(words)-1)]
    n_bigrams = len(bigrams)
    bigram_counts = Counter(bigrams)
    unique_bigrams = len(bigram_counts)
    repeated_bigrams = sum(1 for bg in bigrams if bigram_counts[bg] > 1)
    
    # Trigrams
    trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
    n_trigrams = len(trigrams)
    trigram_counts = Counter(trigrams)
    unique_trigrams = len(trigram_counts)
    repeated_trigrams = sum(1 for tg in trigrams if trigram_counts[tg] > 1)
    
    return {
        'ttr': ttr,
        'repeat_bigram_rate': repeated_bigrams / n_bigrams if n_bigrams > 0 else np.nan,
        'repeat_trigram_rate': repeated_trigrams / n_trigrams if n_trigrams > 0 else np.nan,
        'distinct2': unique_bigrams / n_bigrams if n_bigrams > 0 else np.nan,
        'distinct3': unique_trigrams / n_trigrams if n_trigrams > 0 else np.nan,
    }


# Compute for all essays
print("Computing repetition metrics...")
rep_metrics = []
for _, row in df.iterrows():
    essay_id = row['essay_id']
    text = cohort_texts.get(essay_id, '')
    metrics = compute_repetition_metrics(text)
    metrics['essay_id'] = essay_id
    rep_metrics.append(metrics)

df_rep = pd.DataFrame(rep_metrics)
df = df.merge(df_rep, on='essay_id', how='left')

print(f"\nRepetition metrics computed. Sample:")
print(df[['essay_id', 'score_bin', 'ttr', 'repeat_trigram_rate', 'distinct3']].head())

In [ ]:
# Descriptive stats for repetition metrics by group
REP_METRICS = ['ttr', 'repeat_bigram_rate', 'repeat_trigram_rate', 'distinct2', 'distinct3']

print("="*80)
print("REPETITION METRICS BY GROUP")
print("="*80)

for metric in REP_METRICS:
    print(f"\n--- {metric} ---")
    print(f"{'Group':<15} {'n':>6} {'mean':>10} {'std':>10}")
    print("-"*45)
    for group in GROUP_ORDER:
        g_df = df[df[GROUP_VAR] == group]
        vals = g_df[metric].dropna()
        print(f"{str(group):<15} {len(vals):>6} {vals.mean():>10.4f} {vals.std():>10.4f}")

## 4. Prepare Standardized Variables

In [ ]:
# Standardize control variables
def standardize(s):
    return (s - s.mean()) / s.std()

df['token_count_z'] = standardize(df['token_count'])
df['ppl_W32_z'] = standardize(df['ppl_W32'])
df['repeat_trigram_rate_z'] = standardize(df['repeat_trigram_rate'])
df['distinct3_z'] = standardize(df['distinct3'])
df['ttr_z'] = standardize(df['ttr'])

# Set group as categorical
df[GROUP_VAR] = pd.Categorical(df[GROUP_VAR], categories=GROUP_ORDER, ordered=True)

# Set regime as categorical if exists
if 'regime' in df.columns:
    df['regime'] = pd.Categorical(df['regime'], categories=['short', 'main', 'extended'], ordered=True)

print("Standardized variables created:")
print("  token_count_z, ppl_W32_z, repeat_trigram_rate_z, distinct3_z, ttr_z")

## 5. Regression Models

**Model A:** metric ~ C(group) + token_count_z + C(regime)  
**Model B:** metric ~ C(group) + token_count_z + C(regime) + ppl_W32_z  
**Model C:** metric ~ C(group) + token_count_z + ppl_W32_z + repeat_trigram_rate_z + distinct3_z

In [ ]:
def run_regression_suite(df, metric, group_var):
    """Run Models A, B, C for a given metric."""
    results = {}
    
    # Check if regime exists and has multiple values
    has_regime = 'regime' in df.columns and df['regime'].nunique() > 1
    regime_term = ' + C(regime)' if has_regime else ''
    
    # Model A: Length (+ regime if applicable)
    formula_a = f'{metric} ~ C({group_var}) + token_count_z{regime_term}'
    try:
        model_a = smf.ols(formula_a, data=df.dropna(subset=[metric])).fit()
        results['A'] = {'formula': formula_a, 'model': model_a}
    except Exception as e:
        results['A'] = {'formula': formula_a, 'error': str(e)}
    
    # Model B: + baseline fluency (ppl_W32)
    formula_b = f'{metric} ~ C({group_var}) + token_count_z{regime_term} + ppl_W32_z'
    try:
        model_b = smf.ols(formula_b, data=df.dropna(subset=[metric, 'ppl_W32_z'])).fit()
        results['B'] = {'formula': formula_b, 'model': model_b}
    except Exception as e:
        results['B'] = {'formula': formula_b, 'error': str(e)}
    
    # Model C: + repetition controls
    formula_c = f'{metric} ~ C({group_var}) + token_count_z + ppl_W32_z + repeat_trigram_rate_z + distinct3_z'
    try:
        model_c = smf.ols(formula_c, data=df.dropna(subset=[metric, 'ppl_W32_z', 'repeat_trigram_rate_z', 'distinct3_z'])).fit()
        results['C'] = {'formula': formula_c, 'model': model_c}
    except Exception as e:
        results['C'] = {'formula': formula_c, 'error': str(e)}
    
    return results


# Run for all primary metrics
all_regression_results = {}

print("="*80)
print("REGRESSION RESULTS")
print("="*80)

for metric in PRIMARY_METRICS:
    print(f"\n{'='*70}")
    print(f"METRIC: {metric}")
    print(f"{'='*70}")
    
    results = run_regression_suite(df, metric, GROUP_VAR)
    all_regression_results[metric] = results
    
    for model_name in ['A', 'B', 'C']:
        print(f"\n--- Model {model_name} ---")
        r = results[model_name]
        print(f"Formula: {r['formula']}")
        
        if 'error' in r:
            print(f"ERROR: {r['error']}")
            continue
        
        model = r['model']
        print(f"R²: {model.rsquared:.4f}, n={int(model.nobs)}")
        
        # Show group coefficients
        print(f"\nGroup effects (vs {GROUP_ORDER[0]}):")
        for param, coef in model.params.items():
            if f'C({GROUP_VAR})' in param:
                pval = model.pvalues[param]
                sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
                print(f"  {param}: {coef:+.4f} (p={pval:.4f}) {sig}")

In [ ]:
# Summary comparison table: How do group effects change across models?
print("\n" + "="*80)
print("COEFFICIENT COMPARISON: GROUP EFFECTS ACROSS MODELS")
print("="*80)
print("\nDoes adding fluency/repetition controls reduce the group effect?")
print("(Large reduction = effect was confounded)\n")

# For score experiments, show mid and high vs low
if GROUP_VAR == 'score_bin':
    comparisons = ['mid', 'high']
    
    for comp in comparisons:
        param_name = f"C({GROUP_VAR})[T.{comp}]"
        print(f"\n--- {comp} vs low ---")
        print(f"{'Metric':<20} {'Model A':>12} {'Model B':>12} {'Model C':>12} {'A→C change':>12}")
        print("-"*70)
        
        for metric in PRIMARY_METRICS:
            results = all_regression_results[metric]
            coefs = []
            for model_name in ['A', 'B', 'C']:
                r = results[model_name]
                if 'model' in r and param_name in r['model'].params:
                    coefs.append(r['model'].params[param_name])
                else:
                    coefs.append(np.nan)
            
            if not np.isnan(coefs[0]) and not np.isnan(coefs[2]):
                change_pct = 100 * (coefs[2] - coefs[0]) / abs(coefs[0]) if coefs[0] != 0 else np.nan
                change_str = f"{change_pct:+.0f}%" if not np.isnan(change_pct) else "N/A"
            else:
                change_str = "N/A"
            
            print(f"{metric:<20} {coefs[0]:>12.4f} {coefs[1]:>12.4f} {coefs[2]:>12.4f} {change_str:>12}")

## 6. Visualizations

In [ ]:
# Plot 1: Primary metrics by group (violin)
fig1, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

colors = COLORS.get(EXPERIMENT, {})

for i, metric in enumerate(PRIMARY_METRICS):
    ax = axes[i]
    sns.violinplot(data=df, x=GROUP_VAR, y=metric, order=GROUP_ORDER,
                   palette=colors, ax=ax, inner='box')
    ax.set_xlabel('')
    ax.set_ylabel(metric, fontsize=10)
    ax.set_title(metric, fontsize=11, fontweight='bold')

plt.suptitle(f'Primary Metrics by {GROUP_VAR} ({EXPERIMENT})', fontsize=14, fontweight='bold')
plt.tight_layout()
fig1.savefig(BASE_DIR / EXPERIMENT / 'plots_primary_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 2: Control scatter plots
fig2, axes = plt.subplots(2, 2, figsize=(14, 12))

# 2a: auc_128 vs ppl_W32 (fluency control)
ax = axes[0, 0]
for group in GROUP_ORDER:
    g_df = df[df[GROUP_VAR] == group]
    ax.scatter(g_df['ppl_W32'], g_df['auc_128'], 
               c=colors.get(group, 'gray'), label=str(group), alpha=0.6, s=40)
# Overall regression line
valid = df.dropna(subset=['ppl_W32', 'auc_128'])
z = np.polyfit(valid['ppl_W32'], valid['auc_128'], 1)
p = np.poly1d(z)
x_line = np.linspace(valid['ppl_W32'].min(), valid['ppl_W32'].max(), 100)
ax.plot(x_line, p(x_line), 'k--', alpha=0.7, linewidth=2, label='Overall trend')
ax.set_xlabel('ppl_W32 (baseline fluency)', fontsize=11)
ax.set_ylabel('auc_128', fontsize=11)
ax.set_title('AUC vs Baseline Fluency', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2b: auc_128 vs repeat_trigram_rate (repetition control)
ax = axes[0, 1]
for group in GROUP_ORDER:
    g_df = df[df[GROUP_VAR] == group]
    ax.scatter(g_df['repeat_trigram_rate'], g_df['auc_128'], 
               c=colors.get(group, 'gray'), label=str(group), alpha=0.6, s=40)
valid = df.dropna(subset=['repeat_trigram_rate', 'auc_128'])
z = np.polyfit(valid['repeat_trigram_rate'], valid['auc_128'], 1)
p = np.poly1d(z)
x_line = np.linspace(valid['repeat_trigram_rate'].min(), valid['repeat_trigram_rate'].max(), 100)
ax.plot(x_line, p(x_line), 'k--', alpha=0.7, linewidth=2, label='Overall trend')
ax.set_xlabel('Repeat Trigram Rate', fontsize=11)
ax.set_ylabel('auc_128', fontsize=11)
ax.set_title('AUC vs Repetition', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2c: log_slope_128 vs ppl_W32
ax = axes[1, 0]
for group in GROUP_ORDER:
    g_df = df[df[GROUP_VAR] == group]
    ax.scatter(g_df['ppl_W32'], g_df['log_slope_128'], 
               c=colors.get(group, 'gray'), label=str(group), alpha=0.6, s=40)
valid = df.dropna(subset=['ppl_W32', 'log_slope_128'])
z = np.polyfit(valid['ppl_W32'], valid['log_slope_128'], 1)
p = np.poly1d(z)
x_line = np.linspace(valid['ppl_W32'].min(), valid['ppl_W32'].max(), 100)
ax.plot(x_line, p(x_line), 'k--', alpha=0.7, linewidth=2, label='Overall trend')
ax.set_xlabel('ppl_W32 (baseline fluency)', fontsize=11)
ax.set_ylabel('log_slope_128', fontsize=11)
ax.set_title('Log Slope vs Baseline Fluency', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2d: log_slope_128 vs repeat_trigram_rate
ax = axes[1, 1]
for group in GROUP_ORDER:
    g_df = df[df[GROUP_VAR] == group]
    ax.scatter(g_df['repeat_trigram_rate'], g_df['log_slope_128'], 
               c=colors.get(group, 'gray'), label=str(group), alpha=0.6, s=40)
valid = df.dropna(subset=['repeat_trigram_rate', 'log_slope_128'])
z = np.polyfit(valid['repeat_trigram_rate'], valid['log_slope_128'], 1)
p = np.poly1d(z)
x_line = np.linspace(valid['repeat_trigram_rate'].min(), valid['repeat_trigram_rate'].max(), 100)
ax.plot(x_line, p(x_line), 'k--', alpha=0.7, linewidth=2, label='Overall trend')
ax.set_xlabel('Repeat Trigram Rate', fontsize=11)
ax.set_ylabel('log_slope_128', fontsize=11)
ax.set_title('Log Slope vs Repetition', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(f'Control Variable Relationships ({EXPERIMENT})', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig2.savefig(BASE_DIR / EXPERIMENT / 'plots_controls.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 3: Memory curves (primary windows only)
fig3, ax = plt.subplots(figsize=(10, 6))

for group in GROUP_ORDER:
    g_df = df[df[GROUP_VAR] == group]
    means = [g_df[f'ppl_W{W}'].mean() for W in WINDOWS_PRIMARY]
    ci95 = [1.96 * g_df[f'ppl_W{W}'].sem() for W in WINDOWS_PRIMARY]
    ax.errorbar(WINDOWS_PRIMARY, means, yerr=ci95, marker='o', capsize=4,
                label=f"{group} (n={len(g_df)})", color=colors.get(group), 
                linewidth=2, markersize=8)

ax.set_xlabel('Context Window (tokens)', fontsize=12)
ax.set_ylabel('Perplexity', fontsize=12)
ax.set_title(f'Memory Curves - Primary Windows [32,64,128] ({EXPERIMENT})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xticks(WINDOWS_PRIMARY)

plt.tight_layout()
fig3.savefig(BASE_DIR / EXPERIMENT / 'memory_curves_primary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 4: Extended memory curves (if applicable)
# Check if we have extended window data
has_extended = df['ppl_W512'].notna().sum() > 10

if has_extended:
    fig4, ax = plt.subplots(figsize=(12, 6))
    
    for group in GROUP_ORDER:
        g_df = df[df[GROUP_VAR] == group]
        # Only include essays with all windows
        g_valid = g_df.dropna(subset=[f'ppl_W{W}' for W in WINDOWS_ALL])
        if len(g_valid) < 5:
            continue
        means = [g_valid[f'ppl_W{W}'].mean() for W in WINDOWS_ALL]
        ci95 = [1.96 * g_valid[f'ppl_W{W}'].sem() for W in WINDOWS_ALL]
        ax.errorbar(WINDOWS_ALL, means, yerr=ci95, marker='o', capsize=4,
                    label=f"{group} (n={len(g_valid)})", color=colors.get(group), 
                    linewidth=2, markersize=8)
    
    ax.set_xlabel('Context Window (tokens)', fontsize=12)
    ax.set_ylabel('Perplexity', fontsize=12)
    ax.set_title(f'Memory Curves - Extended Windows [32-512] ({EXPERIMENT})', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(WINDOWS_ALL)
    
    plt.tight_layout()
    fig4.savefig(BASE_DIR / EXPERIMENT / 'memory_curves_extended.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Extended windows not available for most essays. Skipping extended plot.")

## 7. Save All Results

In [ ]:
output_dir = BASE_DIR / EXPERIMENT
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Essay-level results with all new columns
df.to_csv(output_dir / 'essay_level_results.csv', index=False)

# 2. Group summary
summary_rows = []
for group in GROUP_ORDER:
    g_df = df[df[GROUP_VAR] == group]
    row = {'group': group, 'n': len(g_df)}
    for metric in PRIMARY_METRICS + REP_METRICS:
        row[f'{metric}_mean'] = g_df[metric].mean()
        row[f'{metric}_sem'] = g_df[metric].sem()
    summary_rows.append(row)
df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(output_dir / 'group_summary.csv', index=False)

# 3. Regression summaries
with open(output_dir / 'regression_summary.txt', 'w') as f:
    f.write("MODEL A: Length + Regime Control\n")
    f.write("="*70 + "\n\n")
    for metric in PRIMARY_METRICS:
        r = all_regression_results[metric]['A']
        f.write(f"\n{metric}\n{'-'*50}\n")
        f.write(f"Formula: {r['formula']}\n")
        if 'model' in r:
            f.write(r['model'].summary().as_text())
        f.write("\n")

with open(output_dir / 'regression_summary_plus_fluency.txt', 'w') as f:
    f.write("MODEL B: Length + Regime + Baseline Fluency (ppl_W32)\n")
    f.write("="*70 + "\n\n")
    for metric in PRIMARY_METRICS:
        r = all_regression_results[metric]['B']
        f.write(f"\n{metric}\n{'-'*50}\n")
        f.write(f"Formula: {r['formula']}\n")
        if 'model' in r:
            f.write(r['model'].summary().as_text())
        f.write("\n")

with open(output_dir / 'regression_summary_plus_fluency_plus_repetition.txt', 'w') as f:
    f.write("MODEL C: Length + Fluency + Repetition Controls\n")
    f.write("="*70 + "\n\n")
    for metric in PRIMARY_METRICS:
        r = all_regression_results[metric]['C']
        f.write(f"\n{metric}\n{'-'*50}\n")
        f.write(f"Formula: {r['formula']}\n")
        if 'model' in r:
            f.write(r['model'].summary().as_text())
        f.write("\n")

print(f"\nSaved to {output_dir}/")
print(f"  - essay_level_results.csv ({len(df)} essays)")
print(f"  - group_summary.csv")
print(f"  - regression_summary.txt (Model A)")
print(f"  - regression_summary_plus_fluency.txt (Model B)")
print(f"  - regression_summary_plus_fluency_plus_repetition.txt (Model C)")
print(f"  - plots_primary_metrics.png")
print(f"  - plots_controls.png")
print(f"  - memory_curves_primary.png")
if has_extended:
    print(f"  - memory_curves_extended.png")

In [ ]:
# Final summary
print("\n" + "="*80)
print("KEY FINDINGS SUMMARY")
print("="*80)

print("\n1. Do repetition metrics differ by group?")
for metric in ['ttr', 'repeat_trigram_rate', 'distinct3']:
    vals_by_group = [df[df[GROUP_VAR] == g][metric].dropna() for g in GROUP_ORDER]
    if all(len(v) > 2 for v in vals_by_group):
        f_stat, p_val = stats.f_oneway(*vals_by_group)
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
        means = [v.mean() for v in vals_by_group]
        print(f"  {metric}: F={f_stat:.2f}, p={p_val:.4f} {sig}")
        print(f"    Means: {GROUP_ORDER[0]}={means[0]:.4f}, {GROUP_ORDER[-1]}={means[-1]:.4f}")

print("\n2. Do group effects on auc_128/log_slope_128 survive controls?")
for metric in ['auc_128', 'log_slope_128']:
    results = all_regression_results[metric]
    
    # Get high vs low coefficient from each model
    if GROUP_VAR == 'score_bin':
        param = f"C({GROUP_VAR})[T.high]"
        coefs = []
        pvals = []
        for model_name in ['A', 'B', 'C']:
            r = results[model_name]
            if 'model' in r and param in r['model'].params:
                coefs.append(r['model'].params[param])
                pvals.append(r['model'].pvalues[param])
            else:
                coefs.append(np.nan)
                pvals.append(np.nan)
        
        print(f"\n  {metric} (high vs low):")
        print(f"    Model A (length):           β={coefs[0]:+.4f}, p={pvals[0]:.4f}")
        print(f"    Model B (+fluency):         β={coefs[1]:+.4f}, p={pvals[1]:.4f}")
        print(f"    Model C (+fluency+repeat):  β={coefs[2]:+.4f}, p={pvals[2]:.4f}")
        
        if not np.isnan(coefs[0]) and not np.isnan(coefs[2]) and coefs[0] != 0:
            reduction = 100 * (1 - abs(coefs[2]) / abs(coefs[0]))
            print(f"    Effect reduction A→C: {reduction:.1f}%")